# Regiões Hidrográficas do Brasil

Visualização das regiões hidrográficas do Brasil (SNIRH) com destaque para a bacia do Uruguai.

In [ ]:
import shapefile
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection
import numpy as np

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'sans-serif'

In [ ]:
# Carregar shapefile
sf = shapefile.Reader('../data/clusters/bacias/SNIRH_RegioesHidrograficas.shp', encoding='utf-8')
records = sf.records()
shapes = sf.shapes()

print(f'Número de regiões: {len(records)}')
print(f'Campos: {[f.name for f in sf.fields[1:]]}')
print()
for i, rec in enumerate(records):
    print(f"[{i:2d}] {rec['RHI_SG']} - {rec['RHI_NM']:<35} {rec['RHI_AR_KM2']:>12,.0f} km²")

## Plot 1: Todas as Regiões Hidrográficas

In [ ]:
def get_patches(shape, **kwargs):
    """Converte um shape (possivelmente multipolygon) em lista de patches matplotlib."""
    patches = []
    parts = list(shape.parts) + [len(shape.points)]
    for i in range(len(shape.parts)):
        pts = np.array(shape.points[parts[i]:parts[i+1]])
        patch = MplPolygon(pts, closed=True, **kwargs)
        patches.append(patch)
    return patches


def get_centroid(shape):
    """Calcula o centroide aproximado de um shape."""
    pts = np.array(shape.points)
    return pts[:, 0].mean(), pts[:, 1].mean()


# Cores distintas para cada bacia
cmap = plt.colormaps.get_cmap('tab20').resampled(len(shapes))
colors = [cmap(i) for i in range(len(shapes))]

fig, ax = plt.subplots(figsize=(13, 11))

legend_handles = []

for i, (shape, rec) in enumerate(zip(shapes, records)):
    color = colors[i]
    nome = rec['RHI_NM']
    area = rec['RHI_AR_KM2']
    sigla = rec['RHI_SG']

    patches = get_patches(shape)
    for patch in patches:
        patch.set_facecolor(color)
        patch.set_edgecolor('white')
        patch.set_linewidth(0.8)
        ax.add_patch(patch)

    # Sigla no centroide
    cx, cy = get_centroid(shape)
    ax.text(cx, cy, sigla, ha='center', va='center',
            fontsize=9, fontweight='bold', color='white',
            bbox=dict(boxstyle='round,pad=0.15', facecolor=color, edgecolor='none', alpha=0.7))

    handle = mpatches.Patch(facecolor=color, edgecolor='gray', linewidth=0.5,
                            label=f'{sigla} – {nome} ({area:,.0f} km²)')
    legend_handles.append(handle)

# Bbox geral do shapefile
bbox = sf.bbox
margin = 1.0
ax.set_xlim(bbox[0] - margin, bbox[2] + margin)
ax.set_ylim(bbox[1] - margin, bbox[3] + margin)

ax.set_xlabel('Longitude (°)', fontsize=11)
ax.set_ylabel('Latitude (°)', fontsize=11)
ax.set_title('Regiões Hidrográficas do Brasil (SNIRH)', fontsize=14, fontweight='bold', pad=12)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3, linewidth=0.5)

leg = ax.legend(handles=legend_handles, loc='lower left', fontsize=7.5,
                framealpha=0.9, edgecolor='gray', title='Regiões Hidrográficas',
                title_fontsize=8.5)

plt.tight_layout()
plt.savefig('../results/bacias_hidrograficas_brasil.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figura salva em results/bacias_hidrograficas_brasil.png')

## Plot 2: Foco na Bacia do Uruguai

In [ ]:
# Identificar índice da bacia do Uruguai
uruguai_idx = next(i for i, rec in enumerate(records) if rec['RHI_NM'] == 'URUGUAI')
uruguai_shape = shapes[uruguai_idx]
uruguai_rec = records[uruguai_idx]

print(f"Bacia do Uruguai: índice={uruguai_idx}, sigla={uruguai_rec['RHI_SG']}")
print(f"Área: {uruguai_rec['RHI_AR_KM2']:,.0f} km²")
print(f"Bbox: {uruguai_shape.bbox}")

# Cor de destaque
URUGUAI_COLOR = '#2166ac'   # azul
CONTEXT_COLOR = '#d9d9d9'   # cinza claro para as demais
CONTEXT_EDGE  = '#bdbdbd'

fig, ax = plt.subplots(figsize=(10, 9))

# Plotar todas as outras bacias em cinza (contexto)
for i, (shape, rec) in enumerate(zip(shapes, records)):
    if i == uruguai_idx:
        continue
    for patch in get_patches(shape):
        patch.set_facecolor(CONTEXT_COLOR)
        patch.set_edgecolor(CONTEXT_EDGE)
        patch.set_linewidth(0.6)
        ax.add_patch(patch)

# Plotar a bacia do Uruguai com destaque
for patch in get_patches(uruguai_shape):
    patch.set_facecolor(URUGUAI_COLOR)
    patch.set_edgecolor('white')
    patch.set_linewidth(1.2)
    patch.set_zorder(2)
    ax.add_patch(patch)

# Rótulo no centroide da bacia do Uruguai
cx, cy = get_centroid(uruguai_shape)
ax.text(cx, cy, 'URUGUAI', ha='center', va='center',
        fontsize=11, fontweight='bold', color='white', zorder=3,
        bbox=dict(boxstyle='round,pad=0.3', facecolor=URUGUAI_COLOR, edgecolor='none', alpha=0.85))

# Zoom na bacia do Uruguai com margem
margin = 1.0
xmin, ymin, xmax, ymax = uruguai_shape.bbox
ax.set_xlim(xmin - margin, xmax + margin)
ax.set_ylim(ymin - margin, ymax + margin)

ax.set_xlabel('Longitude (°)', fontsize=11)
ax.set_ylabel('Latitude (°)', fontsize=11)
ax.set_title(
    f"Bacia Hidrográfica do Uruguai\n"
    f"Área: {uruguai_rec['RHI_AR_KM2']:,.0f} km²",
    fontsize=13, fontweight='bold', pad=12
)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3, linewidth=0.5)

# Legenda
handles = [
    mpatches.Patch(facecolor=URUGUAI_COLOR, edgecolor='white', label='Bacia do Uruguai'),
    mpatches.Patch(facecolor=CONTEXT_COLOR, edgecolor=CONTEXT_EDGE, label='Demais regiões hidrográficas'),
]
ax.legend(handles=handles, loc='upper right', fontsize=9, framealpha=0.9, edgecolor='gray')

plt.tight_layout()
plt.savefig('../results/bacia_uruguai.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figura salva em results/bacia_uruguai.png')